# FlashAttention from scratch: tile it, stream the softmax, prove it's exact

Standard attention has a dirty secret: its bottleneck is **memory, not math**. To attend over $N$
tokens it builds the full $N \times N$ score matrix and writes it to slow GPU memory (HBM). At
$N = 8192$ that single matrix is 0.125 GiB in fp16 — and you need *one per (batch, head)*, so a
batch of 16 with 32 heads materializes **64 GiB** of scratch: 64 of an 80 GB A100's GiB on attention
intermediates alone, leaving almost nothing for weights and activations. The FLOPs are unchanged;
you drown in memory traffic.

**FlashAttention** ([Dao et al. 2022](https://arxiv.org/abs/2205.14135)) computes the *exact same*
attention output **without ever materializing that matrix**. It tiles K and V into blocks, streams
each block through fast on-chip SRAM, and keeps a **running (online) softmax** — a running max $m$
and running normaliser $\ell$ — rescaling the partial output as it goes.

This notebook builds that algorithm from scratch in small, printable pieces. **By the end you'll have
implemented the online softmax, built blockwise attention on top of it, proven it equals textbook
full-softmax attention to ~1e-6, and watched the running $(m, \ell)$ statistics climb block by block.**

It's a companion to the [KV Cache](../../05-KV-Cache/05-KV-Cache.md) chapter: the KV cache makes
*decode* memory-efficient; FlashAttention makes *prefill and training* IO-efficient. Same enemy —
HBM traffic — attacked from the other side.

## Setup: tiny, real, printable shapes

We use $N = 8$ positions, head dimension $d = 4$, and a block size of $2$ key/value rows per tile —
small enough to print every number, large enough to span 4 blocks. The algorithm is
**device-agnostic**: every tensor is created on `device`, so it runs unchanged on CUDA / MPS / CPU.
We deliberately pin this teaching run to **CPU** so the printed numbers are bit-stable across
machines — and the printed device line says so honestly (swap `device` to the detected accelerator
to run the identical code there). The scale $1/\sqrt{d}$ is the standard attention scaling that keeps
scores out of softmax's saturated region.

In [1]:
import torch
import torch.nn.functional as F

SEQ_LEN = 8       # N: query and key/value positions
HEAD_DIM = 4      # d: dimension of each q/k/v vector
BLOCK_SIZE = 2    # B_c: key/value rows loaded into 'SRAM' per tile
SCALE = HEAD_DIM ** -0.5   # 1/sqrt(d): standard attention scaling
ALLCLOSE_ATOL = 1e-6       # blockwise vs full differ only by float summation-order noise

# Best available accelerator; the algorithm runs on any of these because every tensor is created
# on `device`. We pin THIS run to CPU so the printed numbers are reproducible, and say so honestly.
BEST = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
device = "cpu"   # swap to BEST to run the identical code on the accelerator
print(f"device: {device} (best available: {BEST}; pinned to CPU for reproducibility)")
print("torch:", torch.__version__)

torch.manual_seed(0)
q = torch.randn(SEQ_LEN, HEAD_DIM, device=device)
k = torch.randn(SEQ_LEN, HEAD_DIM, device=device)
v = torch.randn(SEQ_LEN, HEAD_DIM, device=device)
print("q, k, v shapes:", tuple(q.shape), tuple(k.shape), tuple(v.shape))

device: cpu (best available: mps; pinned to CPU for reproducibility)
torch: 2.12.0
q, k, v shapes: (8, 4) (8, 4) (8, 4)


## The baseline FlashAttention must match — and the matrix it refuses to store

Textbook attention forms the full $N \times N$ score matrix $S = QK^\top/\sqrt{d}$, softmaxes each
row over the key axis, and multiplies by $V$. The output is what we have to reproduce. We also
measure how many bytes that score matrix occupies — that's the $O(N^2)$ memory FlashAttention
avoids.

In [2]:
def full_attention(q, k, v):
    """Textbook attention: materialize the whole N x N score matrix, then softmax."""
    scores = (q @ k.transpose(-1, -2)) * SCALE   # (N, N): every query dotted with every key
    weights = F.softmax(scores, dim=-1)          # row-wise softmax over the key axis
    out = weights @ v                            # (N, d): softmax-weighted sum of value rows
    materialized_bytes = scores.numel() * scores.element_size()
    return out, materialized_bytes

out_full, materialized_bytes = full_attention(q, k, v)
print("full attention output shape:", tuple(out_full.shape))
print(f"materialized N x N scores: {materialized_bytes} bytes ({SEQ_LEN}x{SEQ_LEN} floats)")

full attention output shape: (8, 4)
materialized N x N scores: 256 bytes (8x8 floats)


## The heart: online (streaming) softmax

Softmax needs the max and the sum of *all* exponentials to normalise — which seems to force the
whole row into memory at once. The **online softmax** ([Milakov & Gimelshein 2018](https://arxiv.org/abs/1805.02867))
computes it in a single streaming pass instead, carrying two running statistics:

- $m$ — the largest score seen so far (start at $-\infty$),
- $\ell$ — the running sum $\sum \exp(\text{score} - m)$, kept consistent with the current $m$.

When a new block raises the max from $m_\text{old}$ to $m_\text{new}$, every earlier term was
exponentiated against the *wrong* (smaller) max. The fix is one multiply: rescale $\ell$ by
$\exp(m_\text{old} - m_\text{new}) \le 1$, which retroactively re-bases every old term against the
new max. That single correction factor is the log-sum-exp trick made incremental, and it's the
reason the blocks can be combined exactly.

In [3]:
def online_softmax(scores_row, block_size):
    """Softmax of a 1-D row in a genuinely SINGLE streaming pass -- identical to F.softmax.

    Holds only running state: max `m`, denominator `l`, and an un-normalised numerator vector
    `num` (the same kind of accumulator flash_attention uses for its output). When a block raises
    the max, every running quantity is rescaled by exp(m_old - m_new) so all terms end up taken
    against the SAME final max -- that one multiply is what lets the blocks combine exactly.
    """
    n = scores_row.numel()
    running_max = torch.tensor(float("-inf"), device=scores_row.device)   # m
    running_denom = torch.tensor(0.0, device=scores_row.device)           # l
    num = torch.zeros(n, device=scores_row.device)                        # running numerators exp(score - m)
    for start in range(0, n, block_size):
        block = scores_row[start:start + block_size]
        new_max = torch.maximum(running_max, block.max())                 # max across all seen so far
        correction = torch.exp(running_max - new_max)                     # <= 1: shrink prior state to new max
        num = num * correction                                            # re-base every old numerator in one multiply
        num[start:start + block_size] = torch.exp(block - new_max)        # this block's exps, also vs new_max
        running_denom = running_denom * correction + torch.exp(block - new_max).sum()  # rescale denom, add block
        running_max = new_max
    return num / running_denom                                           # single final division

scores_row = (q[0] @ k.transpose(-1, -2)) * SCALE
online = online_softmax(scores_row, BLOCK_SIZE)
reference = F.softmax(scores_row, dim=-1)
softmax_diff = (online - reference).abs().max().item()
print("online softmax :", online.tolist())
print("F.softmax      :", reference.tolist())
assert torch.allclose(online, reference, atol=ALLCLOSE_ATOL)
print(f"\nonline softmax == F.softmax: True   max abs diff: {softmax_diff:.2e}")

online softmax : [0.14138023555278778, 0.06206302344799042, 0.06956938654184341, 0.033915091305971146, 0.07530926167964935, 0.14685700833797455, 0.03339751437306404, 0.4375085234642029]
F.softmax      : [0.14138025045394897, 0.06206303462386131, 0.06956939399242401, 0.033915095031261444, 0.07530925422906876, 0.14685700833797455, 0.03339751437306404, 0.4375085234642029]

online softmax == F.softmax: True   max abs diff: 1.49e-08


## Blockwise attention: fuse the streaming softmax with the value multiply

Now the full algorithm. For each query we walk the key/value blocks **once**, maintaining three
running quantities: the max $m$, the normaliser $\ell$, and an **un-normalised output accumulator**
$\text{acc} = \sum \exp(\text{score} - m)\,\cdot\,\text{value}$. The trick that makes this exact:
when a block raises the max, we rescale **both** $\ell$ **and** $\text{acc}$ by the same
$\exp(m_\text{old} - m_\text{new})$ — the accumulated output is corrected in lockstep with the
denominator. After the last block, $\text{acc}/\ell$ is exact attention, computed with only one
block resident at a time. We also record the running $(m, \ell)$ for query 0 so we can watch them.

In [4]:
def flash_attention(q, k, v, block_size=BLOCK_SIZE):
    """Blockwise attention via a running softmax -- never forms the full N x N matrix."""
    seq_len, head_dim = q.shape
    out = torch.zeros_like(q)
    trace_for_query0 = []
    for i in range(seq_len):                       # one query at a time (kernels tile queries too)
        running_max = torch.tensor(float("-inf"), device=q.device)  # m_i
        running_denom = torch.tensor(0.0, device=q.device)          # l_i
        acc = torch.zeros(head_dim, device=q.device)                # un-normalised output, rescaled as m_i grows
        for block_idx, start in enumerate(range(0, seq_len, block_size)):
            k_block = k[start:start + block_size]  # (B_c, d): key tile in 'SRAM'
            v_block = v[start:start + block_size]  # (B_c, d): matching value tile
            scores = (q[i] @ k_block.transpose(-1, -2)) * SCALE   # this query vs the tile's keys
            new_max = torch.maximum(running_max, scores.max())    # global max so far for query i
            correction = torch.exp(running_max - new_max)         # <= 1: shrink old (l, acc) to new max
            p = torch.exp(scores - new_max)                       # exps of this tile vs the new max
            running_denom = running_denom * correction + p.sum()  # rescale denom, add tile
            acc = acc * correction + p @ v_block                  # rescale output, add weighted values
            running_max = new_max
            if i == 0:
                trace_for_query0.append((block_idx, running_max.item(), running_denom.item()))
        out[i] = acc / running_denom               # final normalisation
    return out, trace_for_query0

out_flash, trace = flash_attention(q, k, v, BLOCK_SIZE)
print("flash attention output shape:", tuple(out_flash.shape))

flash attention output shape: (8, 4)


## Prove it: blockwise output equals full output

This is the whole point. FlashAttention is **exact**, not an approximation — the only difference
from textbook attention is the *order* of summation, which costs a few units in the last place of
floating point. If this assert passes, every optimization that follows is free correctness.

In [5]:
attn_diff = (out_full - out_flash).abs().max().item()
assert torch.allclose(out_full, out_flash, atol=ALLCLOSE_ATOL), "blockwise != full"
print("full  attention row 0:", out_full[0].tolist())
print("flash attention row 0:", out_flash[0].tolist())
print(f"\nflash == full: True   max abs diff over all rows: {attn_diff:.2e}")

full  attention row 0: [-0.6705531477928162, -0.05962146818637848, 0.13087767362594604, 0.5574252605438232]
flash attention row 0: [-0.6705531477928162, -0.05962152034044266, 0.13087761402130127, 0.5574252009391785]

flash == full: True   max abs diff over all rows: 2.09e-07


## Watch the running $(m, \ell)$ climb block by block

Here is the bookkeeping made visible, for query 0. Read it carefully: $m$ is **non-decreasing** (it
only ever rises when a block brings a bigger score). And notice $\ell$ can *drop* between blocks —
that's not a bug, it's the rescaling working. When a later block raises $m$ sharply, the
$\exp(m_\text{old}-m_\text{new})$ factor shrinks the previously accumulated $\ell$ to re-base it
against the new, larger max. Without that correction, the blocks would not combine to the correct
softmax.

In [6]:
print(f"running (max m, denom l) per block for query 0 (block_size={BLOCK_SIZE}):")
for block_idx, m, l in trace:
    print(f"  after block {block_idx}: m = {m:+.4f},  l = {l:.4f}")

running (max m, denom l) per block for query 0 (block_size=2):
  after block 0: m = +0.3350,  l = 1.4390
  after block 1: m = +0.3350,  l = 2.1709
  after block 2: m = +0.3730,  l = 3.6028
  after block 3: m = +1.4647,  l = 2.2857


## The memory argument, in numbers

FlashAttention's win isn't fewer FLOPs — it's fewer bytes moved through HBM. The naive path's
working set is the full $N \times N$ score matrix (grows as $O(N^2)$); FlashAttention's is one
block (independent of $N$). At a realistic context length the difference is the gap between *fits*
and *out-of-memory*.

In [7]:
flash_block_bytes = BLOCK_SIZE * HEAD_DIM * q.element_size()
print(f"naive working set: {materialized_bytes} bytes  ({SEQ_LEN}x{SEQ_LEN} floats) -- O(N^2)")
print(f"flash working set: ~{flash_block_bytes} bytes  ({BLOCK_SIZE}x{HEAD_DIM} floats) -- O(block)")

big_n, batch, heads, fp16, a100_gib = 8192, 16, 32, 2, 80
one_matrix_gib = (big_n * big_n * fp16) / (1024 ** 3)
all_matrices_gib = one_matrix_gib * batch * heads
print(
    f"\nat N={big_n}: one fp16 score matrix = {one_matrix_gib:.3f} GiB; "
    f"x (batch {batch} x {heads} heads) = {all_matrices_gib:.0f} GiB"
)
print(
    f"-> {all_matrices_gib:.0f} of an {a100_gib} GB A100's GiB on attention scratch alone, "
    "leaving almost nothing for weights, activations, and the other layers."
)
print("That is the wall FlashAttention removes.")

naive working set: 256 bytes  (8x8 floats) -- O(N^2)
flash working set: ~32 bytes  (2x4 floats) -- O(block)

at N=8192: one fp16 score matrix = 0.125 GiB; x (batch 16 x 32 heads) = 64 GiB
-> 64 of an 80 GB A100's GiB on attention scratch alone, leaving almost nothing for weights, activations, and the other layers.
That is the wall FlashAttention removes.


## What we built, and what we skipped

**What we proved:**
- The **online softmax** computes the exact softmax in a streaming pass, carrying $(m, \ell)$.
- **Blockwise attention** built on it equals full attention to ~1e-6 — FlashAttention is *exact*.
- The naive working set is $O(N^2)$ scratch in HBM; FlashAttention's is one block — the real win.

**What we skipped** (and where to read it): the **backward pass**, which recomputes the score blocks
on the fly rather than storing them (trading a little extra compute for not holding the $N \times N$
matrix); the **real GPU kernel** that fuses everything into one launch and tiles queries as well as
keys; the **IO-complexity proof** that fewer HBM accesses yield the wall-clock speedup; and the
**variants** — FlashAttention-2/3, sliding-window, and the broader sparse/linear family. All of that
is in the [Efficient Attention — FlashAttention concept page](../06-Efficient-Attention-FlashAttention.md),
with its [curated references](../06-Efficient-Attention-FlashAttention.references.md).

The same verified code lives as a standalone script next to this notebook:
[`flash_attention.py`](flash_attention.py) (run it with `python flash_attention.py`).